# Video inventory audit — 2026-10-10
Read-only profiling of local routing and HF catalog snapshots. File counts do not certify media or physical quality. Live Hugging Face reads returned HTTP 401; no configured credential was available.

In [1]:
from pathlib import Path
import json,gzip,re,collections,hashlib
root=Path.cwd()
if not (root/'docs/joint_split_v3').exists():
    root=next(p for p in root.parents if (p/'docs/joint_split_v3').exists())
parent=root.parent
route_path=root/'docs/joint_split_v3/event_pool_routing.jsonl'
rows=[json.loads(x) for x in route_path.read_text(encoding='utf-8').splitlines() if x.strip()]
cat=parent/'_remote_work/hf_docs_20261010/phyeditingvideo__Phyediting/catalog'
releases=json.loads((cat/'releases.json').read_text(encoding='utf-8'))
with gzip.open(cat/'files.jsonl.gz','rt',encoding='utf-8') as f: files=[json.loads(x) for x in f if x.strip()]
videos=[x for x in files if x['path'].lower().endswith('.mp4')]
by_event=[]
for event in ['T%02d'%i for i in range(1,21)]:
    a=[x for x in rows if x['event_id']==event]
    by_event.append({'event':event,'genesis_videos':sum(x['video_references'] for x in a if x['engine']=='genesis'),'isaac_videos':sum(x['video_references'] for x in a if x['engine']=='isaac'),'pool':a[0]['planned_pool'] if a else 'train_pool','trajectories':len(a)})
groups=collections.defaultdict(set)
for r in rows:groups[r['leakage_group_id']].add(r['planned_pool'])
summary={'scope':'Local snapshots only; no media decoding or remote server QA verification',
 'routing':{'rows':len(rows),'video_references':sum(x['video_references'] for x in rows),'duplicate_record_ids':len(rows)-len({x['record_id'] for x in rows}),'cross_pool_group_conflicts':sum(len(v)>1 for v in groups.values()),'qa_status':dict(collections.Counter(x['qa_status'] for x in rows)),'release_status':dict(collections.Counter(x['release_status'] for x in rows)),'verified_trajectory_records':sum(x['references']['trajectory_file_verified'] for x in rows),'by_event':by_event},
 'main_repo_catalog':{'date':releases['inventory_date'],'revision':releases['source_revision'],'files':len(files),'mp4_paths':len(videos),'unique_mp4_paths':len({x['path'] for x in videos}),'zero_byte_mp4':sum(x['size']==0 for x in videos),'mp4_bytes':sum(x['size'] for x in videos),'release_sum_mp4':sum(x['mp4_files'] for x in releases['releases']),'release_sum_three_view_sets':sum(x['complete_three_view_sets'] for x in releases['releases']),'release_sum_incomplete_camera_sets':sum(x['incomplete_camera_sets'] for x in releases['releases'])}}
summary['catalog_special_cases']={}
for scene in ['T02','T03','T12']:
    rels=[x for x in releases['releases'] if x['scene']==scene]
    vs=[x for x in videos if any(x['path'].startswith(r['prefix']+'/') for r in rels)]
    summary['catalog_special_cases'][scene]={'releases':rels,'example_paths':[x['path'] for x in vs[:4]]}
manifest=json.loads((parent/'scene_preview_20261009/t03_repair_r035/manifest.json').read_text(encoding='utf-8'))
summary['t03_historical_repair']={'samples':len(manifest['samples']),'reasons':dict(collections.Counter(x['reason'] for x in manifest['samples'])),'note':'Repair queue is not a final acceptance receipt.'}
summary['source_sha256']={str(p.relative_to(parent)):hashlib.sha256(p.read_bytes()).hexdigest() for p in [route_path,cat/'releases.json',cat/'files.jsonl.gz']}
output=root/'docs/video_inventory_audit_2026-10-10/audit.json'
output.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))


{
  "scope": "Local snapshots only; no media decoding or remote server QA verification",
  "routing": {
    "rows": 22801,
    "video_references": 68403,
    "duplicate_record_ids": 0,
    "cross_pool_group_conflicts": 0,
    "qa_status": {
      "release_specific_not_reaudited": 16401,
      "ENGINEERING_COMPLETE_VISUAL_QA_PENDING": 2327,
      "VIDEO_FILES_COMPLETE": 1193,
      "visual_qa_pending": 2880
    },
    "release_status": {
      "published_release_qa_not_reaudited": 14474,
      "completed_engineering_export_not_formal_qa_certified": 3520,
      "legacy_scan_not_52800_contract": 4807
    },
    "verified_trajectory_records": 0,
    "by_event": [
      {
        "event": "T01",
        "genesis_videos": 2640,
        "isaac_videos": 960,
        "pool": "train_pool",
        "trajectories": 1200
      },
      {
        "event": "T02",
        "genesis_videos": 2640,
        "isaac_videos": 960,
        "pool": "train_pool",
        "trajectories": 1200
      },
      {
  

In [2]:
# Camera matching supports optional descriptive suffixes after cam01/cam02/cam03.
pattern=re.compile(r'^(.*)__cam(0[123])(?:_[^/]*)?\.mp4$',re.I)
camera_groups=collections.defaultdict(set)
unmatched=[]
for video in videos:
    m=pattern.match(video['path'])
    if m:camera_groups[m[1]].add(m[2])
    else:unmatched.append(video['path'])
correction={}
for event in ['T02','T03','T12']:
    prefixes=[r['prefix']+'/' for r in releases['releases'] if r['scene']==event]
    g={k:v for k,v in camera_groups.items() if any(k.startswith(p) for p in prefixes)}
    correction[event]={'complete_groups':sum(v=={'01','02','03'} for v in g.values()),'incomplete_groups':sum(v!={'01','02','03'} for v in g.values())}
summary['camera_suffix_aware_audit']={'by_event':correction,'total_complete_groups':sum(v=={'01','02','03'} for v in camera_groups.values()),'total_incomplete_groups':sum(v!={'01','02','03'} for v in camera_groups.values()),'unmatched_mp4_count':len(unmatched)}
# Inspect catalogued label/state file types without claiming their contents were validated.
summary['sidecar_catalog_examples']={}
for event in ['T06','T12']:
    prefixes=[r['prefix']+'/' for r in releases['releases'] if r['scene']==event]
    fs=[x['path'] for x in files if any(x['path'].startswith(p) for p in prefixes)]
    nonvideo=[x for x in fs if not x.lower().endswith('.mp4')]
    summary['sidecar_catalog_examples'][event]={'extensions':dict(collections.Counter(Path(x).suffix for x in nonvideo)),'examples':nonvideo[:5]}
output.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps({k:summary[k] for k in ['camera_suffix_aware_audit','sidecar_catalog_examples']},ensure_ascii=False,indent=2))


{
  "camera_suffix_aware_audit": {
    "by_event": {
      "T02": {
        "complete_groups": 884,
        "incomplete_groups": 1
      },
      "T03": {
        "complete_groups": 521,
        "incomplete_groups": 41
      },
      "T12": {
        "complete_groups": 880,
        "incomplete_groups": 0
      }
    },
    "total_complete_groups": 15370,
    "total_incomplete_groups": 42,
    "unmatched_mp4_count": 0
  },
  "sidecar_catalog_examples": {
    "T06": {
      "extensions": {
        ".json": 2641,
        ".gz": 880,
        ".png": 1760
      },
      "examples": [
        "data/T06/production_2640_480p_20260904/BATCH_MANIFEST.json",
        "data/T06/production_2640_480p_20260904/configs/PROD_T06_bg01_home_ase_g01_moon_J_p00__cam01.json",
        "data/T06/production_2640_480p_20260904/configs/PROD_T06_bg01_home_ase_g01_moon_J_p00__cam02.json",
        "data/T06/production_2640_480p_20260904/configs/PROD_T06_bg01_home_ase_g01_moon_J_p00__cam03.json",
        "data/T06/pr